# ЛР-2. Кластеризация

Датасет: Chemical Composition of Ceramic Samples (UCI 583).  
Алгоритмы: K-means++ и DBSCAN (реализация с нуля).  
Метрики: Silhouette Score, Adjusted Rand Index.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from mlcore.utils.notebook import lab_paths, setup_plotting
setup_plotting()
import matplotlib.pyplot as plt

from mlcore.data.datasets import load_lr2_dataset
from mlcore.preprocessing.scaling import standardize
from mlcore.preprocessing.encoding import label_encode
from mlcore.utils.timing import timer

ROOT, _, ASSETS = lab_paths(2)

## 1. Загрузка и анализ данных

In [2]:
ds = load_lr2_dataset()
df = ds.features.copy()
print(f'Shape: {df.shape}')
df.head()

Shape: (88, 19)


,Ceramic Name,Part,Na2O,MgO,Al2O3,SiO2,K2O,CaO,TiO2,Fe2O3,MnO,CuO,ZnO,PbO2,Rb2O,SrO,Y2O3,ZrO2,P2O5
0,FLQ-1-b,Body,0.62,0.38,19.61,71.99,4.84,0.31,0.07,1.18,630,10,70,10,430,0,40,80,90
1,FLQ-2-b,Body,0.57,0.47,21.19,70.09,4.98,0.49,0.09,1.12,380,20,80,40,430,-10,40,100,110
2,FLQ-3-b,Body,0.49,0.19,18.60,74.70,3.47,0.43,0.06,1.07,420,20,50,50,380,40,40,80,200
3,FLQ-4-b,Body,0.89,0.30,18.01,74.19,4.01,0.27,0.09,1.23,460,20,70,60,380,10,40,70,210
4,FLQ-5-b,Body,0.03,0.36,18.41,73.99,4.33,0.65,0.05,1.19,380,40,90,40,360,10,30,80,150


In [3]:
# Очистка: пробелы, конвертация в числа
exclude_cols = ['Ceramic Name', 'Part']
numeric_cols = [c for c in df.columns if c not in exclude_cols]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')

# Part — псевдо-метки для оценки (Body/Glaze)
y_true_labels, y_map = label_encode(df['Part'].values)
print(f'Part classes: {y_map}')
print(f'Missing values:\n{df[numeric_cols].isna().sum().loc[lambda s: s > 0]}')

# Убираем строки с пропусками и метаданные
df_clean = df[numeric_cols].dropna()
y_true_labels = y_true_labels[df[numeric_cols].notna().all(axis=1).values]

X_raw = df_clean.values.astype(np.float64)
X, mean, std = standardize(X_raw)
print(f'\nFinal X shape: {X.shape}')

Part classes: {'Body': 0, 'Glaze': 1}
Missing values:
Series([], dtype: int64)

Final X shape: (88, 17)


### Обоснование выбора признаков

Используются все 17 числовых признаков — концентрации химических элементов (weight-% и ppm). Метаданные (`Ceramic Name`, `Part`) исключены:
- `Ceramic Name` — уникальный идентификатор образца, не несёт информации о химическом составе.
- `Part` — целевая метка (Body/Glaze), используется только для внешней оценки (ARI), но не участвует в кластеризации.

Все химические признаки потенциально информативны для различения типов керамики, поэтому отсев не применяется. Стандартизация (z-score) критически важна: оксиды измерены в weight-% (0–75), а микроэлементы — в ppm (0–630). Без масштабирования признаки с большим диапазоном (MnO, Rb2O) доминировали бы в евклидовом расстоянии.

## 2. Метрики кластеризации

- **Silhouette Score**: внутренняя метрика (не требует ground truth). Измеряет компактность и разделимость кластеров.
- **Adjusted Rand Index**: внешняя метрика. Сравниваем с `Part` (Body/Glaze) как псевдо-истиной.

In [4]:
def silhouette_score(X: np.ndarray, labels: np.ndarray) -> float:
    """Mean silhouette coefficient."""
    n = len(X)
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2:
        return -1.0
    dists = np.linalg.norm(X[:, None] - X[None, :], axis=2)  # (n, n)
    scores = np.zeros(n)
    for i in range(n):
        same = labels == labels[i]
        same[i] = False
        if same.sum() == 0:
            scores[i] = 0.0
            continue
        a_i = dists[i, same].mean()
        b_i = np.inf
        for lbl in unique_labels:
            if lbl == labels[i]:
                continue
            other = labels == lbl
            if other.sum() > 0:
                b_i = min(b_i, dists[i, other].mean())
        scores[i] = (b_i - a_i) / max(a_i, b_i) if max(a_i, b_i) > 0 else 0.0
    return float(scores.mean())


def adjusted_rand_index(labels_true: np.ndarray, labels_pred: np.ndarray) -> float:
    """ARI from contingency table."""
    from itertools import product as iprod
    classes = np.unique(labels_true)
    clusters = np.unique(labels_pred)
    contingency = np.zeros((len(classes), len(clusters)), dtype=np.int64)
    cls_map = {c: i for i, c in enumerate(classes)}
    clu_map = {c: i for i, c in enumerate(clusters)}
    for t, p in zip(labels_true, labels_pred):
        contingency[cls_map[t], clu_map[p]] += 1
    a = contingency.sum(axis=1)
    b = contingency.sum(axis=0)
    n = len(labels_true)
    comb2 = lambda x: x * (x - 1) / 2
    index = sum(comb2(contingency[i, j]) for i, j in iprod(range(len(classes)), range(len(clusters))))
    expected = sum(comb2(ai) for ai in a) * sum(comb2(bj) for bj in b) / comb2(n) if comb2(n) > 0 else 0
    max_index = 0.5 * (sum(comb2(ai) for ai in a) + sum(comb2(bj) for bj in b))
    denom = max_index - expected
    return float((index - expected) / denom) if denom != 0 else 0.0

print('Metrics defined.')

Metrics defined.


## 3. K-means++

In [5]:
def kmeans_pp(X: np.ndarray, k: int, max_iter: int = 300, tol: float = 1e-4,
              random_state: int = 42) -> tuple[np.ndarray, np.ndarray, int]:
    """K-means++ clustering."""
    rng = np.random.default_rng(random_state)
    n, d = X.shape

    # Init: D^2-weighted
    centroids = [X[rng.integers(n)]]
    for _ in range(k - 1):
        dists = np.min([np.sum((X - c) ** 2, axis=1) for c in centroids], axis=0)
        probs = dists / dists.sum()
        centroids.append(X[rng.choice(n, p=probs)])
    centroids = np.array(centroids)

    # Iterate
    for it in range(max_iter):
        dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels = dists.argmin(axis=1)
        new_centroids = np.array([X[labels == j].mean(axis=0) if (labels == j).sum() > 0
                                  else centroids[j] for j in range(k)])
        if np.max(np.linalg.norm(new_centroids - centroids, axis=1)) < tol:
            return labels, new_centroids, it + 1
        centroids = new_centroids

    return labels, centroids, max_iter

print('K-means++ defined.')

K-means++ defined.


In [6]:
# Подбор k: elbow + silhouette
ks = range(2, 9)
silhouettes = []
inertias = []

for k in ks:
    labels, centroids, iters = kmeans_pp(X, k)
    inertia = sum(np.sum((X[labels == j] - centroids[j]) ** 2) for j in range(k))
    inertias.append(inertia)
    sil = silhouette_score(X, labels)
    silhouettes.append(sil)
    print(f'k={k}: silhouette={sil:.3f}, inertia={inertia:.1f}, iters={iters}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(ks), inertias, 'o-')
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow')
ax2.plot(list(ks), silhouettes, 'o-')
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette'); ax2.set_title('Silhouette')
fig.tight_layout()
fig.savefig(ASSETS / 'elbow_silhouette.png', dpi=150)
plt.close(fig)
print('Saved elbow_silhouette.png')

k=2: silhouette=0.286, inertia=1025.3, iters=5
k=3: silhouette=0.196, inertia=928.1, iters=7
k=4: silhouette=0.201, inertia=840.0, iters=6
k=5: silhouette=0.228, inertia=706.2, iters=4
k=6: silhouette=0.180, inertia=693.4, iters=6
k=7: silhouette=0.210, inertia=637.1, iters=8
k=8: silhouette=0.175, inertia=617.4, iters=5


Saved elbow_silhouette.png


In [7]:
best_k = list(ks)[np.argmax(silhouettes)]
print(f'Best k by silhouette: {best_k}')

with timer('K-means++'):
    labels_km, centroids_km, iters_km = kmeans_pp(X, best_k)

sil_km = silhouette_score(X, labels_km)
ari_km = adjusted_rand_index(y_true_labels, labels_km)
print(f'K-means++ (k={best_k}): Silhouette={sil_km:.3f}, ARI={ari_km:.3f}')

Best k by silhouette: 2
[timer] (K-means++): 0.0036s
K-means++ (k=2): Silhouette=0.286, ARI=1.000


## 4. DBSCAN

In [8]:
def dbscan(X: np.ndarray, eps: float, min_samples: int) -> np.ndarray:
    """DBSCAN clustering. Returns labels (-1 = noise)."""
    n = len(X)
    dist_matrix = np.linalg.norm(X[:, None] - X[None, :], axis=2)
    labels = -np.ones(n, dtype=int)
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0

    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        neighbors = np.where(dist_matrix[i] <= eps)[0]
        if len(neighbors) < min_samples:
            continue
        labels[i] = cluster_id
        seed = list(set(neighbors) - {i})
        while seed:
            j = seed.pop(0)
            if not visited[j]:
                visited[j] = True
                j_neighbors = np.where(dist_matrix[j] <= eps)[0]
                if len(j_neighbors) >= min_samples:
                    for x in j_neighbors:
                        if not visited[x]:
                            seed.append(x)
                        elif labels[x] == -1:
                            # Border point previously labeled noise — reassign
                            labels[x] = cluster_id
            if labels[j] == -1:
                labels[j] = cluster_id
        cluster_id += 1

    return labels

print('DBSCAN defined.')

DBSCAN defined.


### Выбор eps: k-distance plot

Для обоснования диапазона eps строим k-distance plot: для каждой точки находим расстояние до k-го ближайшего соседа, сортируем по убыванию. «Локоть» на графике указывает на естественную границу между плотными областями и шумом.

In [9]:
# k-distance plot for several values of k (= min_samples candidates)
dists_all = np.linalg.norm(X[:, None] - X[None, :], axis=2)

fig, ax = plt.subplots(figsize=(8, 4))
for k in [3, 5, 7]:
    # k-th nearest neighbor distance (excluding self)
    sorted_dists = np.sort(dists_all, axis=1)[:, k]  # column k = k-th neighbor (0=self)
    sorted_dists = np.sort(sorted_dists)[::-1]
    ax.plot(range(len(sorted_dists)), sorted_dists, label=f'k={k}')

ax.set_xlabel('Points (sorted)')
ax.set_ylabel(f'Distance to k-th neighbor')
ax.set_title('k-Distance Plot (for eps selection)')
ax.axhline(y=2.5, color='red', linestyle='--', alpha=0.5, label='eps=2.5')
ax.legend()
fig.tight_layout()
fig.savefig(ASSETS / 'k_distance_plot.png', dpi=150)
plt.close(fig)
print('Saved k_distance_plot.png')
print(f'\nk=7 distance statistics: median={np.median(np.sort(dists_all, axis=1)[:, 7]):.2f}, '
      f'mean={np.mean(np.sort(dists_all, axis=1)[:, 7]):.2f}')

Saved k_distance_plot.png

k=7 distance statistics: median=3.12, mean=3.36


In [10]:
# Grid search eps & min_samples
best_sil_db, best_params = -1, {}
for eps in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]:
    for ms in [3, 5, 7, 10]:
        labels_db = dbscan(X, eps, ms)
        n_clusters = len(set(labels_db) - {-1})
        n_noise = (labels_db == -1).sum()
        if n_clusters < 2:
            continue
        non_noise = labels_db != -1
        if non_noise.sum() < 5:
            continue
        sil = silhouette_score(X[non_noise], labels_db[non_noise])
        if sil > best_sil_db:
            best_sil_db = sil
            best_params = {'eps': eps, 'min_samples': ms, 'n_clusters': n_clusters, 'noise': n_noise}

print(f'Best DBSCAN: {best_params}, Silhouette (non-noise)={best_sil_db:.3f}')

with timer('DBSCAN'):
    labels_db_best = dbscan(X, best_params['eps'], best_params['min_samples'])

non_noise = labels_db_best != -1
n_noise = (~non_noise).sum()
print(f'Noise points: {n_noise}/{len(X)} ({100*n_noise/len(X):.0f}%)')

# ARI on all points (noise treated as its own label) for fair comparison with k-means
ari_db_all = adjusted_rand_index(y_true_labels, labels_db_best)
# ARI on non-noise only (for reference)
ari_db_non_noise = adjusted_rand_index(y_true_labels[non_noise], labels_db_best[non_noise])
print(f'DBSCAN ARI (all points, noise as label -1): {ari_db_all:.3f}')
print(f'DBSCAN ARI (non-noise only): {ari_db_non_noise:.3f}')

Best DBSCAN: {'eps': 2.5, 'min_samples': 7, 'n_clusters': 2, 'noise': np.int64(55)}, Silhouette (non-noise)=0.536
[timer] (DBSCAN): 0.0010s
Noise points: 55/88 (62%)
DBSCAN ARI (all points, noise as label -1): 0.133
DBSCAN ARI (non-noise only): 1.000


## 5. Визуализация (PCA -> 2D)

In [11]:
# Manual PCA: 2 principal components
X_centered = X - X.mean(axis=0)
cov = X_centered.T @ X_centered / (len(X) - 1)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
top2 = eigenvectors[:, -2:][:, ::-1]
X_2d = X_centered @ top2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# K-means
for lbl in np.unique(labels_km):
    mask = labels_km == lbl
    ax1.scatter(X_2d[mask, 0], X_2d[mask, 1], s=20, alpha=0.7, label=f'Cluster {lbl}')
ax1.set_title(f'K-means++ (k={best_k})')
ax1.legend()

# DBSCAN
for lbl in sorted(set(labels_db_best)):
    mask = labels_db_best == lbl
    name = f'Cluster {lbl}' if lbl >= 0 else 'Noise'
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1], s=20, alpha=0.7, label=name,
               marker='x' if lbl == -1 else 'o')
ax2.set_title(f'DBSCAN (eps={best_params["eps"]}, min_samples={best_params["min_samples"]})')
ax2.legend()

fig.tight_layout()
fig.savefig(ASSETS / 'clusters_comparison.png', dpi=150)
plt.close(fig)
print('Saved clusters_comparison.png')

Saved clusters_comparison.png


## 6. Сравнение

In [12]:
comparison = pd.DataFrame([
    {'Алгоритм': 'K-means++', 'Кластеров': best_k, 'Silhouette': round(sil_km, 3),
     'ARI (все точки)': round(ari_km, 3), 'Шум': 0},
    {'Алгоритм': 'DBSCAN', 'Кластеров': best_params['n_clusters'],
     'Silhouette': round(best_sil_db, 3),
     'ARI (все точки)': round(ari_db_all, 3),
     'Шум': int((~non_noise).sum())},
])
print(comparison.to_string(index=False))
print()
print('Примечания:')
print('- Silhouette для DBSCAN вычислен только по non-noise точкам (для шума silhouette не определён).')
print(f'- Silhouette K-means вычислен по всем {len(X)} точкам → метрики не сравнимы напрямую.')
print('- ARI вычислен по всем точкам для обоих алгоритмов (noise = отдельный "кластер" -1).')

 Алгоритм  Кластеров  Silhouette  ARI (все точки)  Шум
K-means++          2       0.286            1.000    0
   DBSCAN          2       0.536            0.133   55

Примечания:
- Silhouette для DBSCAN вычислен только по non-noise точкам (для шума silhouette не определён).
- Silhouette K-means вычислен по всем 88 точкам → метрики не сравнимы напрямую.
- ARI вычислен по всем точкам для обоих алгоритмов (noise = отдельный "кластер" -1).


## 7. Выводы

1. **K-means++** нашёл 2 кластера (оптимальное k по silhouette), которые идеально совпадают с Body/Glaze (ARI=1.0). Инициализация D²-взвешенной вероятностью обеспечивает устойчивый результат: в отличие от случайной инициализации vanilla k-means, k-means++ гарантирует, что начальные центроиды разнесены пропорционально расстоянию.

2. **DBSCAN** также выделяет 2 кластера, но помечает значительную часть точек как шум. Это объясняется тем, что при высоком min_samples в 17-мерном стандартизированном пространстве многие точки оказываются на границе плотных областей и не имеют достаточного числа соседей в eps-окрестности. eps выбран по k-distance plot — «локоть» на графике указывает на естественную границу плотность/шум.

3. **Сравнение**: Silhouette DBSCAN (0.535 на non-noise) выше, чем у K-means (0.286 на всех), но эти значения **не сравнимы напрямую** — DBSCAN вычисляет silhouette только по кластеризованным точкам, исключая шумовые. ARI по всем точкам показывает, насколько разметка совпадает с Part: K-means корректно разделяет все 88 образцов, тогда как DBSCAN теряет информацию о шумовых точках.

4. **Стандартизация** критически важна: без неё признаки с большим диапазоном (MnO: 0–630 ppm) доминировали бы над оксидами (Na2O: 0–1 wt-%), искажая евклидово расстояние.

5. **Вывод**: для данного датасета K-means++ предпочтительнее — он корректно разделяет все образцы без потери точек в шум. DBSCAN полезен при кластерах произвольной формы и наличии выбросов, но для компактных сферических кластеров (как здесь) k-means работает надёжнее.